In [1]:
import pandas as pd
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
import statsmodels.api as sms
import re
import seaborn as sns
from sklearn.preprocessing import scale
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.feature_selection import RFE #Recursive feature elimination (RFE)
from statsmodels.stats.stattools import durbin_watson

In [25]:
df = pd.read_csv("car_price.csv")
df.head()

,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0
1,2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0
2,3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0
3,4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0
4,5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0


In [3]:
df["CarName"].unique()

array(['alfa-romero giulia', 'alfa-romero stelvio',
       'alfa-romero Quadrifoglio', 'audi 100 ls', 'audi 100ls',
       'audi fox', 'audi 5000', 'audi 4000', 'audi 5000s (diesel)',
       'bmw 320i', 'bmw x1', 'bmw x3', 'bmw z4', 'bmw x4', 'bmw x5',
       'chevrolet impala', 'chevrolet monte carlo', 'chevrolet vega 2300',
       'dodge rampage', 'dodge challenger se', 'dodge d200',
       'dodge monaco (sw)', 'dodge colt hardtop', 'dodge colt (sw)',
       'dodge coronet custom', 'dodge dart custom',
       'dodge coronet custom (sw)', 'honda civic', 'honda civic cvcc',
       'honda accord cvcc', 'honda accord lx', 'honda civic 1500 gl',
       'honda accord', 'honda civic 1300', 'honda prelude',
       'honda civic (auto)', 'isuzu MU-X', 'isuzu D-Max ',
       'isuzu D-Max V-Cross', 'jaguar xj', 'jaguar xf', 'jaguar xk',
       'maxda rx3', 'maxda glc deluxe', 'mazda rx2 coupe', 'mazda rx-4',
       'mazda glc deluxe', 'mazda 626', 'mazda glc', 'mazda rx-7 gs',
       'mazda glc 

In [26]:
#re comp
p = re.compile(r"\w+-?\w+")
df["car_company"] = df["CarName"].apply(lambda x:re.findall(p,x)[0])

In [27]:
df

,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price,car_company
0,1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,...,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0,alfa-romero
1,2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,...,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0,alfa-romero
2,3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,...,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0,alfa-romero
3,4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,...,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0,audi
4,5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,...,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0,audi
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200,201,-1,volvo 145e (sw),gas,std,four,sedan,rwd,front,109.1,...,mpfi,3.78,3.15,9.5,114,5400,23,28,16845.0,volvo
201,202,-1,volvo 144ea,gas,turbo,four,sedan,rwd,front,109.1,...,mpfi,3.78,3.15,8.7,160,5300,19,25,19045.0,volvo
202,203,-1,volvo 244dl,gas,std,four,sedan,rwd,front,109.1,...,mpfi,3.58,2.87,8.8,134,5500,18,23,21485.0,volvo
203,204,-1,volvo 246,diesel,turbo,four,sedan,rwd,front,109.1,...,idi,3.01,3.40,23.0,106,4800,26,27,22470.0,volvo


In [6]:
df["car_company"].unique()

array(['alfa-romero', 'audi', 'bmw', 'chevrolet', 'dodge', 'honda',
       'isuzu', 'jaguar', 'maxda', 'mazda', 'buick', 'mercury',
       'mitsubishi', 'Nissan', 'nissan', 'peugeot', 'plymouth', 'porsche',
       'porcshce', 'renault', 'saab', 'subaru', 'toyota', 'toyouta',
       'vokswagen', 'volkswagen', 'vw', 'volvo'], dtype=object)

In [8]:
df["corrected_car_company"].unique()

array(['alfa-romeo', 'audi', 'bmw', 'chevrolet', 'dodge', 'honda',
       'isuzu', 'jaguar', 'mazda', 'buick', 'mercury', 'mitsubishi',
       'nissan', 'peugeot', 'plymouth', 'porsche', 'renault', 'saab',
       'subaru', 'toyota', 'volkswagen', 'vw', 'volvo'], dtype=object)

In [20]:
import pandas as pd
import Levenshtein

# Define the correct brand names (reference list)
correct_brands = np.array(['alfa-romeo', 'audi', 'bmw', 'chevrolet', 'dodge', 'honda',
                           'isuzu', 'jaguar', 'mazda', 'buick', 'mercury',
                           'mitsubishi', 'nissan', 'peugeot', 'plymouth', 'porsche',
                           'renault', 'saab', 'subaru', 'toyota', 'volkswagen', 'volvo'])

# Function to find the closest match within the DataFrame itself
def find_closest_match(word, reference_list):
    # Create an empty list to store the results
    result_list = []
    
    # Loop through the reference list and calculate the Levenshtein ratio
    for ref in reference_list:
        score = Levenshtein.ratio(word.lower(), ref.lower())  # Compute similarity ratio
        
        # Append the result for each reference to the result list
        result_list.append({'word': word, 'ref': ref, 'levenstein_score': score})
    
    # Convert the list of results into a DataFrame
    df_result = pd.DataFrame(result_list)
    return df_result

# Test the function with the word 'maxda'
x = 'maxda'
df_output = find_closest_match(x, correct_brands)

# Print the output DataFrame
print(df_output)

     word         ref  levenstein_score
0   maxda  alfa-romeo          0.266667
1   maxda        audi          0.444444
2   maxda         bmw          0.250000
3   maxda   chevrolet          0.000000
4   maxda       dodge          0.200000
5   maxda       honda          0.400000
6   maxda       isuzu          0.000000
7   maxda      jaguar          0.363636
8   maxda       mazda          0.800000
9   maxda       buick          0.000000
10  maxda     mercury          0.166667
11  maxda  mitsubishi          0.133333
12  maxda      nissan          0.181818
13  maxda     peugeot          0.000000
14  maxda    plymouth          0.153846
15  maxda     porsche          0.000000
16  maxda     renault          0.166667
17  maxda        saab          0.444444
18  maxda      subaru          0.181818
19  maxda      toyota          0.181818
20  maxda  volkswagen          0.133333
21  maxda       volvo          0.000000


In [23]:
# Print the output DataFrame
df_output[df_output['levenstein_score']==df_output['levenstein_score'].max()]

,word,ref,levenstein_score
8,maxda,mazda,0.8


In [37]:
import pandas as pd
import numpy as np
import Levenshtein

df1 = df[['car_company']].copy()
# Define the correct brand names (reference list)
correct_brands = np.array(['alfa-romeo', 'audi', 'bmw', 'chevrolet', 'dodge', 'honda',
                           'isuzu', 'jaguar', 'mazda', 'buick', 'mercury',
                           'mitsubishi', 'nissan', 'peugeot', 'plymouth', 'porsche',
                           'renault', 'saab', 'subaru', 'toyota', 'volkswagen', 'volvo'])

# Function to find the best match ratio within the DataFrame
def find_best_score(word, reference_list):
    best_score = 0
    for ref in reference_list:
        score = Levenshtein.ratio(word.lower(), ref.lower())  # Compute similarity ratio
        best_score = max(best_score, score)  # Track the highest similarity score
    return best_score

# Calculate the best score for each word in the DataFrame
df1['best_score'] = df1['car_company'].apply(lambda x: find_best_score(x, correct_brands))

# Calculate the threshold: minimum of the maximum scores
threshold = df1['best_score'].min()

# Function to find the closest match based on the dynamic threshold
def find_closest_match_dynamic(word, reference_list, threshold):
    best_score = 0
    best_match = word
    for ref in reference_list:
        score = Levenshtein.ratio(word.lower(), ref.lower())  # Compute similarity ratio
        if score > best_score:
            best_score = score
            best_match = ref
    return best_match if best_score >= threshold else word  # Use the dynamic threshold

# Apply correction to the DataFrame column based on the dynamic threshold
df1["corrected_car_company"] = df1["car_company"].apply(lambda x: find_closest_match_dynamic(x, correct_brands, threshold))

# Display the DataFrame
print("Dynamic Threshold:", threshold)

Dynamic Threshold: 0.4


In [40]:
# Group by car_company and aggregate the best_score and corrected_car_company
grouped_df = df.groupby('car_company').agg(
    best_score=('best_score', 'max'),  # Take the maximum best_score for each car_company
    corrected_car_company=('corrected_car_company', 'first')  # Use the first corrected value
).reset_index()

# Display the grouped result
grouped_df

,car_company,best_score,corrected_car_company
0,Nissan,1.000000,nissan
1,alfa-romero,0.952381,alfa-romeo
2,audi,1.000000,audi
3,bmw,1.000000,bmw
4,buick,1.000000,buick
5,chevrolet,1.000000,chevrolet
6,dodge,1.000000,dodge
7,honda,1.000000,honda
8,isuzu,1.000000,isuzu
9,jaguar,1.000000,jaguar
